---
title: "Data Collection"
---


{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

In [1]:
import json
import os
import random
import time
import requests

from pathlib import Path
from dotenv import find_dotenv, load_dotenv

In [2]:
API_BASE = "https://api.ravelry.com"

RAW_DIR = Path("../../data/raw-data")
RAW_PATH = RAW_DIR / "ravelry_patterns.jsonl"
MANIFEST_PATH = RAW_DIR / "ravelry_manifest.json"

TARGET_N = 30000           
BATCH_SIZE = 50          
PAUSE_SECONDS = 1.0      
MAX_RETRIES = 5
SEED = 5000
MAX_ID = 1_500_000  

DROP_FIELDS = ("photos", "notes_html")
RETRY_STATUSES = {429, 500, 502, 503, 504}

In [3]:
def make_session():
    load_dotenv(find_dotenv(usecwd=True))
    username = os.getenv("RAVELRY_USERNAME")
    password = os.getenv("RAVELRY_PASSWORD")

    if not username or not password:
        raise RuntimeError("username or password not set. pls set in env")
    session = requests.session()
    session.auth = (username,password)
    session.headers["User-Agent"] = "DSAN 5000 student project (Georgetown University)"
    return session

In [4]:
def get_json(session, path, params):
    for attempt in range(MAX_RETRIES):
        response = session.get(f"{API_BASE}{path}", params=params, timeout=30)
        if response.status_code in RETRY_STATUSES:
            time.sleep(PAUSE_SECONDS * 2 **(attempt +1))
            continue
        if response.status_code == 404:
            return None
        response.raise_for_status()
        time.sleep(PAUSE_SECONDS)
        return response.json()
    raise RuntimeError(f"Gave up on {path} after {MAX_RETRIES} retries")

In [5]:
def save_manifest(manifest: dict) -> None:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

def load_or_create_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    manifest = {"max_id": MAX_ID, "seed": SEED, "next_index": 0}
    save_manifest(manifest)
    return manifest

def shuffled_ids(max_id: int, seed: int) -> list[int]:
    ids = list(range(1, max_id + 1))
    random.Random(seed).shuffle(ids)
    return ids

In [6]:
def fetch_patterns(session, ids):
    if not ids:
        return []
    data = get_json(session, "/patterns.json", {"ids": " ".join(map(str, ids))})
    if data is not None:
        return [p for p in data.get("patterns", {}).values() if p]
    if len(ids) == 1:
        return []
    mid = len(ids) // 2
    return fetch_patterns(session, ids[:mid]) + fetch_patterns(session, ids[mid:])

In [7]:
def trim_record(pattern: dict) -> dict:
    return {k: v for k, v in pattern.items() if k not in DROP_FIELDS}


def append_records(records: list[dict]) -> None:
    with RAW_PATH.open("a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")


def count_saved() -> int:
    if not RAW_PATH.exists():
        return 0
    with RAW_PATH.open(encoding="utf-8") as f:
        return sum(1 for _ in f)

In [ ]:
def collect(session: requests.Session) -> None:
    manifest = load_or_create_manifest()
    candidates = shuffled_ids(manifest["max_id"], manifest["seed"])
    saved = count_saved()
    start = time.time()

    while saved < TARGET_N and manifest["next_index"] < len(candidates):
        i = manifest["next_index"]
        batch = candidates[i : i + BATCH_SIZE]
        records = [trim_record(p) for p in fetch_patterns(session, batch)]
        append_records(records)
        saved += len(records)
        manifest["next_index"] = i + len(batch)
        save_manifest(manifest)

        if (i // BATCH_SIZE) % 20 == 0:
            hit_rate = saved / max(manifest["next_index"], 1)
            print(f"{saved:,}/{TARGET_N:,} saved | {manifest['next_index']:,} IDs tried "
                  f"| hit rate {hit_rate:.0%} | {(time.time() - start) / 60:.1f} min")

    print(f"Done: {saved:,} patterns in {RAW_PATH}")


session = make_session()
collect(session)

886/30,000 saved | 1,050 IDs tried | hit rate 84% | 6.8 min
1,734/30,000 saved | 2,050 IDs tried | hit rate 85% | 14.4 min
2,555/30,000 saved | 3,050 IDs tried | hit rate 84% | 22.6 min
3,404/30,000 saved | 4,050 IDs tried | hit rate 84% | 30.5 min
4,231/30,000 saved | 5,050 IDs tried | hit rate 84% | 39.0 min
5,067/30,000 saved | 6,050 IDs tried | hit rate 84% | 47.4 min
5,917/30,000 saved | 7,050 IDs tried | hit rate 84% | 55.1 min
6,768/30,000 saved | 8,050 IDs tried | hit rate 84% | 62.6 min
7,615/30,000 saved | 9,050 IDs tried | hit rate 84% | 70.1 min
8,455/30,000 saved | 10,050 IDs tried | hit rate 84% | 78.2 min
9,283/30,000 saved | 11,050 IDs tried | hit rate 84% | 86.3 min
10,123/30,000 saved | 12,050 IDs tried | hit rate 84% | 94.3 min
10,950/30,000 saved | 13,050 IDs tried | hit rate 84% | 102.6 min
11,782/30,000 saved | 14,050 IDs tried | hit rate 84% | 111.0 min
12,620/30,000 saved | 15,050 IDs tried | hit rate 84% | 119.0 min
13,461/30,000 saved | 16,050 IDs tried | hit 

In [ ]:
def summarize_raw() -> None:
    crafts, free, sample = {}, 0, None
    with RAW_PATH.open(encoding="utf-8") as f:
        for n, line in enumerate(f, start=1):
            p = json.loads(line)
            craft = (p.get("craft") or {}).get("name", "Unknown")
            crafts[craft] = crafts.get(craft, 0) + 1
            free += bool(p.get("free"))
            sample = sample or p
    print(f"{n:,} patterns | {free / n:.0%} free")
    print("By craft:", dict(sorted(crafts.items(), key=lambda kv: -kv[1])))
    print("Fields available:", sorted(sample.keys()))


summarize_raw()

172 patterns | 25% free
By craft: {'Knitting': 103, 'Crochet': 65, 'Machine Knitting': 3, 'Loom Knitting': 1}
Fields available: ['comments_count', 'craft', 'created_at', 'currency', 'currency_symbol', 'difficulty_average', 'difficulty_count', 'download_location', 'downloadable', 'favorites_count', 'free', 'gauge', 'gauge_description', 'gauge_divisor', 'gauge_pattern', 'gauge_repeats', 'generally_available', 'has_uk_terminology', 'has_us_terminology', 'id', 'languages', 'name', 'notes', 'packs', 'pattern_attributes', 'pattern_author', 'pattern_categories', 'pattern_needle_sizes', 'pattern_type', 'pdf_in_library', 'pdf_url', 'permalink', 'personal_attributes', 'price', 'printings', 'product_id', 'projects_count', 'published', 'queued_projects_count', 'rating_average', 'rating_count', 'ravelry_download', 'row_gauge', 'sizes_available', 'unlisted_product_ids', 'updated_at', 'url', 'volumes_in_library', 'yardage', 'yardage_description', 'yardage_max', 'yarn_list_type', 'yarn_weight', 'yarn_

{{< include closing.qmd >}} 